In [1]:
"""
ibge_coleta.py
==============
Coleta os endpoints IBGE de prioridade Alta definidos em:
  fontes_covariaveis_municipais.md

Fontes cobertas:
  1. IBGE Localidades   — /api/v1/localidades/municipios
  2. IBGE Censo 2022    — SIDRA tabelas 9606, 9605, 9514  (via apisidra)
"""

from locale import D_FMT
import time
import requests
import pandas as pd

# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────

def get_json(url: str, params: dict = None, label: str = "") -> dict | list | None:
    """GET com tratamento de erro e log simples."""
    try:
        resp = requests.get(url, params=params, timeout=90)
        resp.raise_for_status()
        print(f"  ✔ {label or url}")
        return resp.json()
    except requests.HTTPError as e:
        print(f"  ✘ {label} — HTTP {e.response.status_code}")
    except Exception as e:
        print(f"  ✘ {label} — {e}")
    return None


# ══════════════════════════════════════════════
# 1. IBGE LOCALIDADES
# ══════════════════════════════════════════════

def coletar_localidades() -> pd.DataFrame:
    """
    Retorna todos os ~5.570 municípios com:
    id, nome, uf_sigla, uf_nome, regiao_imediata,
    regiao_intermediaria, macroregiao_sigla
    """
    print("\n[1/3] Coletando Localidades…")
    url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
    data = get_json(url, label="municipios")
    if not data:
        return pd.DataFrame()

    df = pd.DataFrame([
        {
            "id_municipio":         m["id"],
            "nome_municipio":       m["nome"],
            "uf_sigla":             m["regiao-imediata"]["regiao-intermediaria"]["UF"]["sigla"],
            "uf_nome":              m["regiao-imediata"]["regiao-intermediaria"]["UF"]["nome"],
            "regiao_imediata_id":   m["regiao-imediata"]["id"],
            "regiao_imediata_nome": m["regiao-imediata"]["nome"],
            "regiao_interm_id":     m["regiao-imediata"]["regiao-intermediaria"]["id"],
            "regiao_interm_nome":   m["regiao-imediata"]["regiao-intermediaria"]["nome"],
            "macroregiao_sigla":    m["regiao-imediata"]["regiao-intermediaria"]["UF"]["regiao"]["sigla"],
            "macroregiao_nome":     m["regiao-imediata"]["regiao-intermediaria"]["UF"]["regiao"]["nome"],
        }
        for m in data
    ])
    print(f"     → {len(df)} municípios carregados")
    return df


# ══════════════════════════════════════════════
# 2. IBGE CENSO 2022 — SIDRA
# ══════════════════════════════════════════════

# Tabelas definidas no documento:
#   9606 — Domicílios com internet por município
#   9605 — Rendimento médio domiciliar per capita
#   9514 — População por sexo e faixa etária
#
# Endpoint padrão apisidra:
#   /values/t/{tabela}/n6/all/v/all/p/last

SIDRA_BASE = "https://apisidra.ibge.gov.br/values"

TABELAS_CENSO = {
    "9606": "Domicilios_com_internet",
    "9605": "Rendimento_medio_domiciliar_percapita",
    "9514": "Populacao_sexo_faixa_etaria",
}


def _parse_sidra(data: list) -> pd.DataFrame:
    """
    A API SIDRA retorna uma lista onde o índice 0 é o cabeçalho
    e os demais são registros.
    """
    if not data or len(data) < 2:
        return pd.DataFrame()
    header = list(data[0].keys())
    columns_names = list(data[0].values())
    rows   = data[1:]
    df = pd.DataFrame(rows, columns=header)
    df.columns = columns_names

    return df


def coletar_censo_sidra() -> dict[str, pd.DataFrame]:
    """
    Retorna um dict  {nome_tabela: DataFrame}
    para cada tabela do Censo 2022 listada acima.
    """
    print("\n[2/3] Coletando Censo 2022 via SIDRA…")
    resultados = {}

    for tabela, nome in TABELAS_CENSO.items():
        url = f"{SIDRA_BASE}/t/{tabela}/n6/all/v/all/p/last"
        data = get_json(url, label=f"SIDRA tabela {tabela} — {nome}")

        if data:
            df = _parse_sidra(data)
            print(f"     → {len(df)} linhas")
            resultados[nome] = df
        else:
            resultados[nome] = pd.DataFrame()

        time.sleep(0.5)   # respeita rate-limit da API

    return resultados


In [19]:
localidades = coletar_localidades()
localidades.head(1)



[1/3] Coletando Localidades…
  ✔ municipios
     → 5571 municípios carregados


,id_municipio,nome_municipio,uf_sigla,uf_nome,regiao_imediata_id,regiao_imediata_nome,regiao_interm_id,regiao_interm_nome,macroregiao_sigla,macroregiao_nome
0,1100015,Alta Floresta D'Oeste,RO,Rondônia,110005,Cacoal,1102,Ji-Paraná,N,Norte


In [3]:
import json

# ── JSON bruto — Localidades ──────────────────────────────────────────────────
_raw_loc = get_json(
    "https://servicodados.ibge.gov.br/api/v1/localidades/municipios",
    label="municipios (raw)",
)

print(f"Tipo da resposta : {type(_raw_loc)}")
print(f"Total de itens   : {len(_raw_loc)}")
print(f"Tipo de cada item: {type(_raw_loc[0])}")
print(f"Chaves de nível 1: {list(_raw_loc[0].keys())}")
print("\n── Primeiro item — JSON bruto completo ──")
print(json.dumps(_raw_loc[0], ensure_ascii=False, indent=2))

  ✔ municipios (raw)
Tipo da resposta : <class 'list'>
Total de itens   : 5571
Tipo de cada item: <class 'dict'>
Chaves de nível 1: ['id', 'nome', 'microrregiao', 'regiao-imediata']

── Primeiro item — JSON bruto completo ──
{
  "id": 1100015,
  "nome": "Alta Floresta D'Oeste",
  "microrregiao": {
    "id": 11006,
    "nome": "Cacoal",
    "mesorregiao": {
      "id": 1102,
      "nome": "Leste Rondoniense",
      "UF": {
        "id": 11,
        "sigla": "RO",
        "nome": "Rondônia",
        "regiao": {
          "id": 1,
          "sigla": "N",
          "nome": "Norte"
        }
      }
    }
  },
  "regiao-imediata": {
    "id": 110005,
    "nome": "Cacoal",
    "regiao-intermediaria": {
      "id": 1102,
      "nome": "Ji-Paraná",
      "UF": {
        "id": 11,
        "sigla": "RO",
        "nome": "Rondônia",
        "regiao": {
          "id": 1,
          "sigla": "N",
          "nome": "Norte"
        }
      }
    }
  }
}


In [6]:
import json, time

# ── JSON bruto — SIDRA (data[0] = cabeçalho, data[1] = primeiro registro) ────

for tabela, nome in TABELAS_CENSO.items():
    url = f"{SIDRA_BASE}/t/{tabela}/n6/all/v/all/p/last"
    _raw = get_json(url, label=f"SIDRA {tabela} (raw)")

    print(f"\n{'═'*60}")
    print(f"  Tabela {tabela} — {nome}")
    print(f"{'═'*60}")
    print(f"Total de itens (cabeçalho + registros): {len(_raw)}")
    print(f"Tipo de cada item                     : {type(_raw[0])}")

    print("\n── data[0] — mapa de cabeçalho (chave interna → nome legível) ──")
    print(json.dumps(_raw[0], ensure_ascii=False, indent=2))

    print("\n── data[1] — primeiro registro real (chaves internas) ──")
    print(json.dumps(_raw[1], ensure_ascii=False, indent=2))

    time.sleep(0.5)

  ✔ SIDRA 9606 (raw)

════════════════════════════════════════════════════════════
  Tabela 9606 — Domicilios_com_internet
════════════════════════════════════════════════════════════
Total de itens (cabeçalho + registros): 11141
Tipo de cada item                     : <class 'dict'>

── data[0] — mapa de cabeçalho (chave interna → nome legível) ──
{
  "NC": "Nível Territorial (Código)",
  "NN": "Nível Territorial",
  "MC": "Unidade de Medida (Código)",
  "MN": "Unidade de Medida",
  "V": "Valor",
  "D1C": "Município (Código)",
  "D1N": "Município",
  "D2C": "Variável (Código)",
  "D2N": "Variável",
  "D3C": "Ano (Código)",
  "D3N": "Ano",
  "D4C": "Sexo (Código)",
  "D4N": "Sexo",
  "D5C": "Cor ou raça (Código)",
  "D5N": "Cor ou raça",
  "D6C": "Idade (Código)",
  "D6N": "Idade"
}

── data[1] — primeiro registro real (chaves internas) ──
{
  "NC": "6",
  "NN": "Município",
  "MC": "45",
  "MN": "Pessoas",
  "V": "21494",
  "D1C": "1100015",
  "D1N": "Alta Floresta D'Oeste - RO",
  "D

In [16]:
resultados = coletar_censo_sidra()


[2/3] Coletando Censo 2022 via SIDRA…
  ✔ SIDRA tabela 9606 — Domicilios_com_internet
     → 11140 linhas
  ✔ SIDRA tabela 9605 — Rendimento_medio_domiciliar_percapita
     → 11140 linhas
  ✔ SIDRA tabela 9514 — Populacao_sexo_faixa_etaria
     → 11140 linhas


In [20]:
resultados['Domicilios_com_internet'].head(1)

,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano,Sexo (Código),Sexo,Cor ou raça (Código),Cor ou raça,Idade (Código),Idade
0,6,Município,45,Pessoas,21494,1100015,Alta Floresta D'Oeste - RO,93,População residente,2022,2022,6794,Total,95251,Total,100362,Total


In [21]:
resultados['Rendimento_medio_domiciliar_percapita'].head(1)

,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano,Cor ou raça (Código),Cor ou raça
0,6,Município,45,Pessoas,21494,1100015,Alta Floresta D'Oeste - RO,93,População residente,2022,2022,95251,Total


In [22]:
resultados['Populacao_sexo_faixa_etaria'].head(1)

,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano,Sexo (Código),Sexo,Forma de declaração da idade (Código),Forma de declaração da idade,Idade (Código),Idade
0,6,Município,45,Pessoas,21494,1100015,Alta Floresta D'Oeste - RO,93,População residente,2022,2022,6794,Total,113635,Total,100362,Total


In [ ]:
# ══════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════

if __name__ == "__main__":

    # 1. Localidades ──────────────────────────
    df_localidades = coletar_localidades()
    print(df_localidades.head(3).to_string())

    # 2. Censo 2022 / SIDRA ───────────────────
    censo = coletar_censo_sidra()
    for nome, df in censo.items():
        print(f"\n── {nome} ({len(df)} linhas) ──")
        print(df.head(3).to_string())

# Testando e aprendendo 

In [12]:
import os
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import pandas as pd
import requests
import structlog
from pydantic import BaseModel
from tenacity import retry, retry_if_exception, stop_after_attempt, wait_exponential

log = structlog.get_logger()

_TABELAS = ["9606", "9605", "9514"]
_TIMEOUT = 60
_SLEEP_ENTRE_CHAMADAS = 0.5
_VOLUME_MINIMO = 11000


In [13]:


class SidraRegistroRaw(BaseModel):
    NC: str
    NN: str
    MC: str
    MN: str
    V: Optional[str]
    D1C: str
    D1N: str
    D2C: str
    D2N: str
    D3C: str
    D3N: str
    D4C: str
    D4N: str
    D5C: Optional[str] = None
    D5N: Optional[str] = None
    D6C: Optional[str] = None
    D6N: Optional[str] = None


In [15]:
url = f"https://apisidra.ibge.gov.br/values/t/9606/n6/all/v/all/p/last"
data = get_json(url)
# data[5].get("V")
data[5]


  ✔ https://apisidra.ibge.gov.br/values/t/9606/n6/all/v/all/p/last


{'NC': '6',
 'NN': 'Município',
 'MC': '45',
 'MN': 'Pessoas',
 'V': '5351',
 'D1C': '1100031',
 'D1N': 'Cabixi - RO',
 'D2C': '93',
 'D2N': 'População residente',
 'D3C': '2022',
 'D3N': '2022',
 'D4C': '6794',
 'D4N': 'Total',
 'D5C': '95251',
 'D5N': 'Total',
 'D6C': '100362',
 'D6N': 'Total'}

In [16]:


records = []
for row in data[1:]:
    v_raw = row.get("V")
    v = None if v_raw in ("-", "...") else v_raw
    records.append(SidraRegistroRaw(
            NC=row["NC"],
            NN=row["NN"],
            MC=row["MC"],
            MN=row["MN"],
            V=v,
            D1C=row["D1C"],
            D1N=row["D1N"],
            D2C=row["D2C"],
            D2N=row["D2N"],
            D3C=row["D3C"],
            D3N=row["D3N"],
            D4C=row["D4C"],
            D4N=row["D4N"],
            D5C=row.get("D5C"),
            D5N=row.get("D5N"),
            D6C=row.get("D6C"),
            D6N=row.get("D6N"),
        ))

In [ ]:
records[1]

SidraRegistroRaw(NC='6', NN='Município', MC='2', MN='Percentual', V='100.00', D1C='1100015', D1N="Alta Floresta D'Oeste - RO", D2C='1000093', D2N='População residente - percentual do total geral', D3C='2022', D3N='2022', D4C='6794', D4N='Total', D5C='95251', D5N='Total', D6C='100362', D6N='Total')

In [ ]:
records[1].model_dump()

{'NC': '6',
 'NN': 'Município',
 'MC': '2',
 'MN': 'Percentual',
 'V': '100.00',
 'D1C': '1100015',
 'D1N': "Alta Floresta D'Oeste - RO",
 'D2C': '1000093',
 'D2N': 'População residente - percentual do total geral',
 'D3C': '2022',
 'D3N': '2022',
 'D4C': '6794',
 'D4N': 'Total',
 'D5C': '95251',
 'D5N': 'Total',
 'D6C': '100362',
 'D6N': 'Total'}

In [21]:
pd.DataFrame(list(records[1].model_dump()))

,0
0,NC
1,NN
2,MC
3,MN
4,V
5,D1C
6,D1N
7,D2C
8,D2N
9,D3C


In [ ]:
# def _gravar(records: list[SidraRegistroRaw], tabela: str, today: datetime) -> None:
    # df = pd.DataFrame([r.model_dump() for r in records])
    # base = os.environ.get("RAW_BASE_PATH", "data/raw")
    # dest = (
    #     Path(base)
    #     / f"ibge_censo_{tabela}"
    #     / f"year={today.year}"
    #     / f"month={today.month:02d}"
    #     / f"day={today.day:02d}"
    #     / "data.parquet"
    # )
    # dest.parent.mkdir(parents=True, exist_ok=True)
    # df.to_parquet(dest, index=False, compression="snappy")
    # log.info("ibge_censo.parquet_gravado", tabela=tabela, destino_path=str(dest))

In [1]:
import os
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

import pandas as pd
import requests
import structlog
from pydantic import BaseModel
from tenacity import retry, retry_if_exception, stop_after_attempt, wait_exponential

log = structlog.get_logger()

_TABELAS = ["9606", "9605", "9514"]
_TIMEOUT = 60
_SLEEP_ENTRE_CHAMADAS = 0.5
_VOLUME_MINIMO = 11000


In [2]:


class SidraRegistroRaw(BaseModel):
    NC: str
    NN: str
    MC: str
    MN: str
    V: Optional[str]
    D1C: str
    D1N: str
    D2C: str
    D2N: str
    D3C: str
    D3N: str
    D4C: str
    D4N: str
    D5C: Optional[str] = None
    D5N: Optional[str] = None
    D6C: Optional[str] = None
    D6N: Optional[str] = None


SidraRegistroRaw

__main__.SidraRegistroRaw

In [3]:


def _retryable(exc: BaseException) -> bool:
    if isinstance(exc, requests.HTTPError):
        return exc.response.status_code >= 500
    return isinstance(exc, (requests.ConnectionError, requests.Timeout))



In [ ]:

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception(_retryable),
    reraise=True,
)
def _fetch(tabela: str) -> list[dict]:
    url = f"https://apisidra.ibge.gov.br/values/t/{tabela}/n6/all/v/all/p/last"
    r = requests.get(url, timeout=_TIMEOUT)
    r.raise_for_status()
    return r.json()


def _parse_sidra(data: list[dict]) -> list[SidraRegistroRaw]:
    # data[0] é o mapa de cabeçalho — ignorar
    records = []
    for row in data[1:]:
        v_raw = row.get("V")
        v = None if v_raw in ("-", "...") else v_raw
        records.append(SidraRegistroRaw(
            NC=row["NC"],
            NN=row["NN"],
            MC=row["MC"],
            MN=row["MN"],
            V=v,
            D1C=row["D1C"],
            D1N=row["D1N"],
            D2C=row["D2C"],
            D2N=row["D2N"],
            D3C=row["D3C"],
            D3N=row["D3N"],
            D4C=row["D4C"],
            D4N=row["D4N"],
            D5C=row.get("D5C"),
            D5N=row.get("D5N"),
            D6C=row.get("D6C"),
            D6N=row.get("D6N"),
        ))
    return records


def _gravar(records: list[SidraRegistroRaw], tabela: str, today: datetime) -> None:
    df = pd.DataFrame([r.model_dump() for r in records])
    base = os.environ.get("RAW_BASE_PATH", "data/raw")
    dest = (
        Path(base)
        / f"ibge_censo_{tabela}"
        / f"year={today.year}"
        / f"month={today.month:02d}"
        / f"day={today.day:02d}"
        / "data.parquet"
    )
    dest.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(dest, index=False, compression="snappy")
    log.info("ibge_censo.parquet_gravado", tabela=tabela, destino_path=str(dest))


def run() -> None:
    today = datetime.now(tz=timezone.utc)
    for i, tabela in enumerate(_TABELAS):
        if i > 0:
            time.sleep(_SLEEP_ENTRE_CHAMADAS)
        log.info("ibge_censo.inicio", tabela=tabela)
        try:
            data = _fetch(tabela)
            records = _parse_sidra(data)
            log.info("ibge_censo.fetch_ok", tabela=tabela, total_registros=len(records))
            if len(records) < _VOLUME_MINIMO:
                log.warning("ibge_censo.volume_baixo", tabela=tabela, total=len(records), minimo_esperado=_VOLUME_MINIMO)
            _gravar(records, tabela, today)
        except Exception:
            log.error("ibge_censo.erro_fatal", tabela=tabela, exc_info=True)
            raise  # write parcial é pior que sem write — aborta o run inteiro


if __name__ == "__main__":
    run()


## Testes 

In [2]:
MUNICIPIO_COMPLETO = {
    "id": 5200050,
    "nome": "Abadia de Goiás",
    "regiao-imediata": {
        "id": 52008,
        "nome": "Goiânia",
        "regiao-intermediaria": {
            "id": 5203,
            "nome": "Goiânia",
            "UF": {
                "id": 52,
                "sigla": "GO",
                "nome": "Goiás",
                "regiao": {"id": 5, "sigla": "CO", "nome": "Centro-Oeste"},
            },
        },
    },
    "microrregiao": {
        "id": 52010,
        "nome": "Goiânia",
        "mesorregiao": {"id": 5203, "nome": "Centro Goiano"},
    },
}

In [5]:
# {print(f"{i}:{k}") for i,k in MUNICIPIO_COMPLETO.items()}
{i:k for i,k in MUNICIPIO_COMPLETO.items()}

{'id': 5200050,
 'nome': 'Abadia de Goiás',
 'regiao-imediata': {'id': 52008,
  'nome': 'Goiânia',
  'regiao-intermediaria': {'id': 5203,
   'nome': 'Goiânia',
   'UF': {'id': 52,
    'sigla': 'GO',
    'nome': 'Goiás',
    'regiao': {'id': 5, 'sigla': 'CO', 'nome': 'Centro-Oeste'}}}},
 'microrregiao': {'id': 52010,
  'nome': 'Goiânia',
  'mesorregiao': {'id': 5203, 'nome': 'Centro Goiano'}}}

In [26]:
[MUNICIPIO_COMPLETO] * 10

[{'id': 5200050,
  'nome': 'Abadia de Goiás',
  'regiao-imediata': {'id': 52008,
   'nome': 'Goiânia',
   'regiao-intermediaria': {'id': 5203,
    'nome': 'Goiânia',
    'UF': {'id': 52,
     'sigla': 'GO',
     'nome': 'Goiás',
     'regiao': {'id': 5, 'sigla': 'CO', 'nome': 'Centro-Oeste'}}}},
  'microrregiao': {'id': 52010,
   'nome': 'Goiânia',
   'mesorregiao': {'id': 5203, 'nome': 'Centro Goiano'}}},
 {'id': 5200050,
  'nome': 'Abadia de Goiás',
  'regiao-imediata': {'id': 52008,
   'nome': 'Goiânia',
   'regiao-intermediaria': {'id': 5203,
    'nome': 'Goiânia',
    'UF': {'id': 52,
     'sigla': 'GO',
     'nome': 'Goiás',
     'regiao': {'id': 5, 'sigla': 'CO', 'nome': 'Centro-Oeste'}}}},
  'microrregiao': {'id': 52010,
   'nome': 'Goiânia',
   'mesorregiao': {'id': 5203, 'nome': 'Centro Goiano'}}},
 {'id': 5200050,
  'nome': 'Abadia de Goiás',
  'regiao-imediata': {'id': 52008,
   'nome': 'Goiânia',
   'regiao-intermediaria': {'id': 5203,
    'nome': 'Goiânia',
    'UF': {'id

In [7]:
SIDRA_HEADER = {
    "NC": "Nível Territorial (Código)",
    "NN": "Nível Territorial",
    "MC": "Unidade de Medida (Código)",
    "MN": "Unidade de Medida",
    "V": "Valor",
    "D1C": "Município (Código)",
    "D1N": "Município",
    "D2C": "Variável (Código)",
    "D2N": "Variável",
    "D3C": "Ano (Código)",
    "D3N": "Ano",
    "D4C": "Dimensão 4 (Código)",
    "D4N": "Dimensão 4",
}

SIDRA_RECORD = {
    "NC": "6",
    "NN": "Município",
    "MC": "Pessoa",
    "MN": "Pessoa",
    "V": "21494",
    "D1C": "1100015",
    "D1N": "Alta Floresta D'Oeste - RO",
    "D2C": "93",
    "D2N": "População residente",
    "D3C": "2022",
    "D3N": "2022",
    "D4C": "6561",
    "D4N": "Total",
}

In [9]:
data = [SIDRA_HEADER, SIDRA_RECORD, SIDRA_RECORD]
len(data) -1 


2

In [48]:
from unittest.mock import MagicMock, patch

magic_mock = MagicMock()

<MagicMock name='mock.maca' id='137746187330896'>

In [32]:
magic_mock.status_code = 505

In [40]:
e= requests.HTTPError(response = magic_mock)

In [36]:
e#.response.status_code

requests.exceptions.HTTPError()

In [37]:
mock_resp = MagicMock()
mock_resp.status_code = 404
mock_resp.raise_for_status.side_effect = requests.HTTPError(response=mock_resp)

try:
    mock_resp.raise_for_status()
except requests.HTTPError as e:
    print(e.response.status_code)  # 404

404


In [47]:
type(mock_resp.raise_for_status)

unittest.mock.MagicMock

In [43]:
e = requests.HTTPError(response=mock_resp)
type(e)

requests.exceptions.HTTPError

o requests.HTTPError(response=mock_resp) é um objeto requests.exceptions.HTTPError que está sendo para o atributo de mock_resp side_effect? 

In [ ]:

e.response.status_code

In [35]:
try:
    magic_mock.raise_for_status()
except requests.HTTPError(response = magic_mock) as e: 
    print(e.response.status_code)

In [28]:
magic_mock.raise_for_status.side_effect

In [14]:
data[2]

{'NC': '6',
 'NN': 'Município',
 'MC': 'Pessoa',
 'MN': 'Pessoa',
 'V': '21494',
 'D1C': '1100015',
 'D1N': "Alta Floresta D'Oeste - RO",
 'D2C': '93',
 'D2N': 'População residente',
 'D3C': '2022',
 'D3N': '2022',
 'D4C': '6561',
 'D4N': 'Total'}

In [15]:
_parse_sidra (data)

,Nível Territorial (Código),Nível Territorial,Unidade de Medida (Código),Unidade de Medida,Valor,Município (Código),Município,Variável (Código),Variável,Ano (Código),Ano,Dimensão 4 (Código),Dimensão 4
0,6,Município,Pessoa,Pessoa,21494,1100015,Alta Floresta D'Oeste - RO,93,População residente,2022,2022,6561,Total
1,6,Município,Pessoa,Pessoa,21494,1100015,Alta Floresta D'Oeste - RO,93,População residente,2022,2022,6561,Total


In [27]:
import requests

url = f"https://apisidra.ibge.gov.br/values/t/9606/n6/all/v/all/p/last"
r = requests.get(url)


In [ ]:
data = r.json()


In [59]:
data[0]

{'NC': 'Nível Territorial (Código)',
 'NN': 'Nível Territorial',
 'MC': 'Unidade de Medida (Código)',
 'MN': 'Unidade de Medida',
 'V': 'Valor',
 'D1C': 'Município (Código)',
 'D1N': 'Município',
 'D2C': 'Variável (Código)',
 'D2N': 'Variável',
 'D3C': 'Ano (Código)',
 'D3N': 'Ano',
 'D4C': 'Sexo (Código)',
 'D4N': 'Sexo',
 'D5C': 'Cor ou raça (Código)',
 'D5N': 'Cor ou raça',
 'D6C': 'Idade (Código)',
 'D6N': 'Idade'}

In [33]:
data[1].get('V')

'21494'

In [43]:
def retorna_ibge_pessoas(table = 9606): 

    response = requests.get(f"https://apisidra.ibge.gov.br/values/t/{table}/n6/all/v/all/p/last")

    return response.json()[1].get('V')


In [47]:
retorna_ibge_pessoas(table = 9606)

'21494'

In [60]:
from unittest import mock
from unittest.mock import patch, MagicMock

mock_response = MagicMock()
mock_response.json.return_value = [{"V": "pessoas"}, {"V": 99.90}]

In [63]:
with patch("requests.get", return_value = mock_response): 
    pessoas = retorna_ibge_pessoas(table = 9606)

    print (pessoas)

99.9
